# Episode 32: Events

Telemetry, observers, audit logs — every one of them in Group 6 was built on the same thing: the **event stream**. This episode goes to the source.

**In this episode you'll build:** direct programs against the stream — counting events, filtering on field values, and *changing* events in flight.

## The stream is the substrate

Every agent run is a sequence of events flowing through a `MemoryStream`: the model is asked, a tool is called, a result comes back, a message is produced. Observers and telemetry are just *subscribers* on that stream.

Pass your own `MemoryStream` to `ask(stream=...)` and you can subscribe before the run starts — `stream.where(EventType).subscribe()` registers a handler. Everything else in this episode builds on that one call.

## Every run is a sequence of events

Subscribe to *all* events and tally them by type. A single tool-using turn produces a rich, structured trace — request, response, tool call, tool result, final message.

In [1]:
from dotenv import load_dotenv

load_dotenv()

from collections import Counter

from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."


agent = Agent("weather_bot", prompt="Use get_weather for weather questions.",
              config=config, tools=[get_weather])

seen: Counter = Counter()
stream = MemoryStream()


@stream.subscribe()
def tally(event) -> None:
    seen[type(event).__name__] += 1


reply = await agent.ask("What's the weather in Paris?", stream=stream)

print("event counts:")
for name, n in sorted(seen.items()):
    print(f"  {name}: {n}")


event counts:
  ModelMessage: 1
  ModelRequest: 1
  ModelResponse: 2
  ToolCallEvent: 1
  ToolCallsEvent: 1
  ToolResultEvent: 1
  ToolResultsEvent: 1


## Filtering: type, OR, and field value

`where()` is the filter. Pass an event type, combine types with `|`, or match on a **field value** with a comparison — `ToolCallEvent.name == "get_weather"`.

Below, one subscriber fires for *every* tool call; the other fires only for `get_weather`. The agent has two tools, so the difference is visible.

In [2]:
from autogen.beta.events import ToolCallEvent


def get_time(city: str) -> str:
    """Get the current time in a city."""
    return f"{city}: 14:00."


bot = Agent("bot", prompt="Use get_weather and get_time as needed.",
            config=config, tools=[get_weather, get_time])

stream = MemoryStream()
any_tool_calls: list[str] = []
weather_calls: list[str] = []


@stream.where(ToolCallEvent).subscribe()
def any_tool(event: ToolCallEvent) -> None:
    any_tool_calls.append(event.name)


@stream.where(ToolCallEvent.name == "get_weather").subscribe()
def weather_only(event: ToolCallEvent) -> None:
    weather_calls.append(event.name)


reply = await bot.ask("What's the weather and time in Paris?", stream=stream)

print("any-tool subscriber saw:    ", sorted(any_tool_calls))
print("weather-only subscriber saw:", weather_calls)
print("reply:", reply.body)


any-tool subscriber saw:     ['get_time', 'get_weather']
weather-only subscriber saw: ['get_weather']
reply: The current weather in Paris is 21°C and sunny. The current time in Paris is 14:00.


## Interrupters: changing events in flight

A normal subscriber only *observes*. An **interrupter** — `subscribe(interrupt=True)` — runs before regular subscribers and can **modify** the event: return a changed event and that version flows on; return `None` and the event is suppressed entirely.

Here an interrupter rewrites every `ModelMessage` to uppercase. The agent is told to reply in lowercase — but the interrupter has the last word, so the final reply comes out shouting.

In [3]:
from autogen.beta.events import ModelMessage

agent = Agent("quiet_bot", prompt="Reply in lowercase. Be brief.", config=config)

stream = MemoryStream()


@stream.where(ModelMessage).subscribe(interrupt=True)
def shout(event: ModelMessage) -> ModelMessage:
    event.content = event.content.upper()
    return event


reply = await agent.ask("say hello to the workshop", stream=stream)
print("reply:", reply.body)


reply: HELLO, WORKSHOP!


## Other stream tools

Two more ways to work with the stream:

```python
# Subscribe only inside a block — auto-unsubscribed on exit.
with stream.where(ModelMessage).sub_scope(handler):
    ...

# Await the next occurrence of an event.
async with stream.get(HumanMessage) as future:
    event = await future
```

`sub_scope` is the scoped form of `subscribe`; `get` lets you *wait* for an event rather than react to it — useful for human-in-the-loop checkpoints.

## Additional content

**Observers are stream subscribers.** Episode 25's `@observer` and `BaseObserver` are just subscribers registered on the agent instead of the stream. Same machinery — observers are the ergonomic wrapper.

**Raise events from your own code.** Inside a tool or subscriber, `await context.send(event)` publishes an event onto the stream. Always go through `context` — it keeps stream scope and dependency injection intact.

**Interrupters are powerful — and sharp.** Modifying or suppressing events changes what the agent and the user see. Reach for a normal subscriber unless you specifically need to alter the run.

## What's next

You now understand AG2 Beta end to end. Episode 33 is the **migration guide** — moving real code from AG2 Classic, AutoGen 0.x, and LangChain onto everything you've learned.